# Step 4 — Does the window probability actually work? (calibration, enrichment, stability, baselines)

**Click Runtime ▸ Run all.** Nothing to upload. Expect 30–50 minutes on free Colab.

This notebook tests the paper's central claim with held-out data:
1. **Calibration:** when the model says "30% chance the gap is in 1.0–1.8 eV", is it right about 30% of the time?
2. **Enrichment:** if you pick the top 5% of compounds by this probability, how many more truly-in-window compounds do you get than by picking at random, or by the naive "closest to 1.4 eV" ranking?
3. **Stability:** do the main numbers change when the cross-validation split changes (3 random seeds)?
4. **Baselines:** random forest vs gradient boosting vs ridge for the gap regressor.

If any cell turns red, **stop and send me the last 5 lines.**

## Cell 1 — Install
If Colab shows *Restart session*, click it, then Run all again.

In [ ]:
%pip install -q matminer pymatgen scikit-learn pandas numpy scipy matplotlib

## Cell 2 — Load data and features

In [ ]:
import os, re, json, math, shutil, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

SEED = 42
OUT = "step4_outputs"
os.makedirs(OUT, exist_ok=True)

from matminer.datasets import load_dataset
raw = load_dataset("matbench_expt_gap")
target_col = [c for c in raw.columns if "gap" in str(c).lower()][0]
formula_col = [c for c in raw.columns if c != target_col][0]
df = pd.DataFrame({"formula": raw[formula_col].astype(str), "gap": raw[target_col].astype(float)})
print("Rows:", len(df), "(expected 4604)")
assert len(df) > 4000, "Dataset looks wrong - stop and send me this message."

TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)")
def parse_formula(f):
    comp = {}
    for el, n in TOKEN.findall(f):
        comp[el] = comp.get(el, 0.0) + (float(n) if n else 1.0)
    return comp
def chem_system(f): return "-".join(sorted(parse_formula(f)))
def family(f):
    els = set(parse_formula(f))
    if "O" in els: return "oxide"
    if els & {"S","Se","Te"}: return "chalcogenide"
    if els & {"F","Cl","Br","I"}: return "halide"
    if els & {"N","P","As","Sb"}: return "pnictide"
    return "other"

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
try:
    EP = ElementProperty.from_preset("magpie", impute_nan=True)
except TypeError:
    EP = ElementProperty.from_preset("magpie")
def featurize(formulas):
    rows = [EP.featurize(Composition(parse_formula(f))) for f in formulas]
    return np.nan_to_num(np.array(rows, dtype=float))

df = df[df.formula.apply(lambda f: len(parse_formula(f)) > 0)].reset_index(drop=True)
df["system"] = df.formula.apply(chem_system); df["family"] = df.formula.apply(family)
y = df.gap.values; nm = (y > 0).astype(int); groups = df.system.values; fams = df.family.values
WIN_LO, WIN_HI = 1.0, 1.8
truth = (y >= WIN_LO) & (y <= WIN_HI)
print("Computing features (1-2 minutes)...")
X = featurize(df.formula.tolist())
print("Features:", X.shape, "| share of compounds truly in the 1.0-1.8 eV window: %.3f" % truth.mean())

N_FOLDS = 5
def grouped_folds(groups, seed):
    rng = np.random.default_rng(seed)
    cnt = Counter(groups); keys = list(cnt); rng.shuffle(keys); keys.sort(key=lambda k: -cnt[k])
    load = np.zeros(N_FOLDS); assign = {}
    for k in keys:
        j = int(np.argmin(load)); assign[k] = j; load[j] += cnt[k]
    fo = np.array([assign[g] for g in groups])
    return [(np.where(fo != j)[0], np.where(fo == j)[0]) for j in range(N_FOLDS)]
def rf_reg(n=200): return RandomForestRegressor(n_estimators=n, n_jobs=-1, random_state=SEED)
def rf_clf(n=200): return RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=SEED)
def p_nonmetal(clf, Xq):
    return clf.predict_proba(Xq)[:, list(clf.classes_).index(1)] if 1 in clf.classes_ else np.zeros(len(Xq))
def cluster_mean_ci(vals, clusters, n_boot=2000, seed=SEED):
    vals = np.asarray(vals, float); u, inv = np.unique(clusters, return_inverse=True)
    s = np.bincount(inv, weights=vals); c = np.bincount(inv); rng = np.random.default_rng(seed); k = len(u); b = np.empty(n_boot)
    for i in range(n_boot):
        pick = rng.integers(0, k, k); b[i] = s[pick].sum() / c[pick].sum()
    return float(vals.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

## Cell 3 — Stability: main numbers across 3 different cross-validation splits

In [ ]:
SEEDS = [42, 43, 44]
stab = []
for sd in SEEDS:
    single = np.zeros(len(df)); regp = np.zeros(len(df)); pnm = np.zeros(len(df))
    for tr, te in grouped_folds(groups, sd):
        assert not (set(groups[tr]) & set(groups[te])), "leak"
        single[te] = rf_reg().fit(X[tr], y[tr]).predict(X[te])
        trn = tr[nm[tr] == 1]
        regp[te] = rf_reg().fit(X[trn], y[trn]).predict(X[te])
        pnm[te] = p_nonmetal(rf_clf().fit(X[tr], nm[tr]), X[te])
    hard = np.where(pnm > 0.5, regp, 0.0); m1 = nm == 1
    stab.append(dict(seed=sd, MAE_single=np.abs(y - single).mean(), MAE_two_stage=np.abs(y - hard).mean(),
                     MAE_nonmetal_single=np.abs(y - single)[m1].mean(), MAE_nonmetal_two_stage=np.abs(y - hard)[m1].mean(),
                     MAE_nonmetal_regressor=np.abs(y - regp)[m1].mean(), accuracy=((pnm > 0.5).astype(int) == nm).mean()))
    print("seed", sd, "done", flush=True)
ST = pd.DataFrame(stab).round(4); ST.to_csv(f"{OUT}/stability_by_seed.csv", index=False)
print(ST.to_string(index=False))
summ = ST.drop(columns="seed").agg(["mean", "std", "min", "max"]).round(4)
summ.to_csv(f"{OUT}/stability_summary.csv"); print("\n", summ.to_string())

## Cell 4 — Calibration and enrichment of the window probability
For each outer fold, the training part is split by chemical system into a *proper-training* set (80%) and a *calibration* set (20%). Models are fit on the proper-training set; the calibration residuals turn each prediction into a probability; the held-out fold is only used for scoring. A compound is *truly in window* if its measured gap is within 1.0–1.8 eV (metals, gap 0, are not).

In [ ]:
def split_cal(tr, seed, frac=0.2):
    rng = np.random.default_rng(seed); g = groups[tr]; uniq = np.unique(g); rng.shuffle(uniq)
    cnt = Counter(g); target = frac * len(tr); acc = 0; chosen = set()
    for u in uniq:
        if acc >= target: break
        chosen.add(u); acc += cnt[u]
    m = np.array([x in chosen for x in g]); return tr[~m], tr[m]

RUNS = {}
for sd in SEEDS:
    pw = np.zeros(len(df)); pn_all = np.zeros(len(df)); pr_all = np.zeros(len(df)); ps_all = np.zeros(len(df))
    for k, (tr, te) in enumerate(grouped_folds(groups, sd)):
        ptr, cal = split_cal(tr, sd * 100 + k)
        assert not (set(groups[cal]) & set(groups[ptr])) and not (set(groups[te]) & set(groups[tr])), "leak"
        ptr_nm = ptr[nm[ptr] == 1]; cal_nm = cal[nm[cal] == 1]
        clf = rf_clf().fit(X[ptr], nm[ptr]); reg = rf_reg().fit(X[ptr_nm], y[ptr_nm]); sng = rf_reg().fit(X[ptr], y[ptr])
        r = y[cal_nm] - reg.predict(X[cal_nm])
        pn = p_nonmetal(clf, X[te]); pr = reg.predict(X[te]); t = pr[:, None] + r[None, :]
        pw[te] = pn * ((t >= WIN_LO) & (t <= WIN_HI)).mean(axis=1)
        pn_all[te] = pn; pr_all[te] = pr; ps_all[te] = sng.predict(X[te])
    RUNS[sd] = dict(pw=pw, pn=pn_all, pr=pr_all, ps=ps_all)
    print("calibration run, seed", sd, "done", flush=True)

def brier(p, t): return float(np.mean((p - t) ** 2))
def prec_at(score, t, q):
    n = len(t); k = max(1, int(math.ceil(q * n))); order = np.argsort(-score, kind="stable")[:k]; return float(t[order].mean())
def boot_enrich(score, t, cl, q, n_boot=500, seed=SEED):
    rng = np.random.default_rng(seed); u = np.unique(cl); idx = {c: np.where(cl == c)[0] for c in u}; out = []
    for _ in range(n_boot):
        ii = np.concatenate([idx[c] for c in rng.choice(u, len(u), replace=True)])
        base = t[ii].mean()
        if base > 0: out.append(prec_at(score[ii], t[ii], q) / base)
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

subsets = {"all compounds": np.ones(len(df), bool), "chalcogenides + pnictides": np.isin(fams, ["chalcogenide", "pnictide"])}
rows = []
for sd, R in RUNS.items():
    scores = {"window probability (ours)": R["pw"],
              "naive: single-stage closest to 1.4 eV": -np.abs(R["ps"] - 1.4),
              "naive: two-stage closest to 1.4 eV": np.where(R["pn"] > 0.5, -np.abs(R["pr"] - 1.4), -99.0)}
    for sname, mk in subsets.items():
        t = truth[mk]; base = float(t.mean())
        for mname, sc in scores.items():
            for q in (0.02, 0.05, 0.10):
                pr_ = prec_at(sc[mk], t, q)
                rows.append(dict(seed=sd, subset=sname, ranking=mname, top_fraction=q, base_rate=round(base, 3),
                                 precision=round(pr_, 3), enrichment=round(pr_ / base, 2) if base > 0 else np.nan))
EN = pd.DataFrame(rows); EN.to_csv(f"{OUT}/enrichment_by_seed.csv", index=False)
agg = (EN.groupby(["subset", "ranking", "top_fraction"])[["base_rate", "precision", "enrichment"]].agg(["mean", "std"]).round(3))
agg.to_csv(f"{OUT}/enrichment_summary.csv"); print(agg.to_string())

cal_rows = []
for sd, R in RUNS.items():
    for sname, mk in subsets.items():
        p = R["pw"][mk]; t = truth[mk].astype(float)
        cal_rows.append(dict(seed=sd, subset=sname, base_rate=t.mean(), brier=brier(p, t), brier_constant_base_rate=brier(np.full_like(p, t.mean()), t),
                             brier_skill_score=1 - brier(p, t) / brier(np.full_like(p, t.mean()), t),
                             AUC_in_window=roc_auc_score(t, p), mean_predicted_probability=p.mean()))
CAL = pd.DataFrame(cal_rows).round(4); CAL.to_csv(f"{OUT}/calibration_scores.csv", index=False); print("\n", CAL.to_string(index=False))

# CI for enrichment at 5% (seed 42, system-clustered bootstrap)
ci_rows = []
R = RUNS[SEEDS[0]]
for sname, mk in subsets.items():
    lo, hi = boot_enrich(R["pw"][mk], truth[mk], groups[mk], 0.05)
    ci_rows.append(dict(subset=sname, top_fraction=0.05, enrichment_ci95_low=round(lo, 2), enrichment_ci95_high=round(hi, 2)))
CI = pd.DataFrame(ci_rows); CI.to_csv(f"{OUT}/enrichment_ci_seed42.csv", index=False); print("\n", CI.to_string(index=False))

# reliability table + figure (seed 42)
fig, axs = plt.subplots(1, 2, figsize=(9, 3.9)); REL = []
for ax, (sname, mk) in zip(axs, subsets.items()):
    p = R["pw"][mk]; t = truth[mk].astype(float)
    qs = np.unique(np.quantile(p, np.linspace(0, 1, 9))); bins = np.digitize(p, qs[1:-1])
    xs, ys, ns = [], [], []
    for b in np.unique(bins):
        s = bins == b; xs.append(p[s].mean()); ys.append(t[s].mean()); ns.append(int(s.sum()))
        REL.append(dict(subset=sname, bin=int(b), n=int(s.sum()), mean_predicted=round(float(p[s].mean()), 3), observed_fraction=round(float(t[s].mean()), 3)))
    ax.plot([0, max(xs + ys) * 1.05], [0, max(xs + ys) * 1.05], "k--", lw=.8); ax.plot(xs, ys, "o-")
    ax.set_title(sname); ax.set_xlabel("Predicted probability of 1.0-1.8 eV"); ax.set_ylabel("Observed fraction")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_reliability.png", dpi=150); plt.show()
pd.DataFrame(REL).to_csv(f"{OUT}/reliability_table_seed42.csv", index=False); print(pd.DataFrame(REL).to_string(index=False))

# enrichment figure
sub = EN[(EN.subset == "chalcogenides + pnictides") & (EN.seed == SEEDS[0])]
fig, ax = plt.subplots(figsize=(6, 3.6)); w = 0.26
for i, rk in enumerate(sub.ranking.unique()):
    s = sub[sub.ranking == rk]; ax.bar(np.arange(3) + (i - 1) * w, s.enrichment.values, w, label=rk)
ax.axhline(1, color="k", lw=.8); ax.set_xticks(range(3)); ax.set_xticklabels(["top 2%", "top 5%", "top 10%"])
ax.set_ylabel("Enrichment over random picking"); ax.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_enrichment.png", dpi=150); plt.show()

## Cell 5 — Baselines: which regressor for non-metal gaps?
Same chemical-system folds (seed 42), trained on non-metals only; scored on true non-metals.

In [ ]:
models = {"Random forest": lambda: rf_reg(),
          "Gradient boosting": lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=SEED),
          "Ridge": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0))}
folds0 = grouped_folds(groups, SEEDS[0]); rows = []
for name, mk in models.items():
    pr = np.zeros(len(df))
    for tr, te in folds0:
        trn = tr[nm[tr] == 1]; pr[te] = mk().fit(X[trn], y[trn]).predict(X[te])
    m1 = nm == 1; mae, lo, hi = cluster_mean_ci(np.abs(y - pr)[m1], groups[m1])
    rows.append(dict(model=name, MAE_nonmetals_eV=round(mae, 3), ci_low=round(lo, 3), ci_high=round(hi, 3)))
    print(name, "done", flush=True)
BASE = pd.DataFrame(rows); BASE.to_csv(f"{OUT}/regressor_baselines.csv", index=False); print(BASE.to_string(index=False))

## Cell 6 — Save and download

In [ ]:
summary = dict(window=[WIN_LO, WIN_HI], base_rate_all=float(truth.mean()), seeds=SEEDS,
               stability=ST.to_dict(orient="records"), stability_summary=summ.reset_index().to_dict(orient="records"),
               calibration=CAL.to_dict(orient="records"), enrichment_ci_seed42=CI.to_dict(orient="records"),
               enrichment_summary=EN.groupby(["subset", "ranking", "top_fraction"])[["base_rate", "precision", "enrichment"]].mean().round(3).reset_index().to_dict(orient="records"),
               baselines=BASE.to_dict(orient="records"))
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2, default=float)
shutil.make_archive("step4_results", "zip", OUT)
print("Saved step4_results.zip")
try:
    from google.colab import files
    files.download("step4_results.zip")
except ImportError:
    pass

## When it finishes
Send me `summary.json` from `step4_results.zip` (or paste the printed output of Cells 3, 4 and 5). Whatever it shows — good or bad — goes into the paper honestly.